In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # points to project root from notebooks/

In [2]:
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
import os

from src.pipeline.predict import load_model, DIGIT_LABELS
from src.evaluation.eval_utils import run_image, build_gt_boxes_map

In [3]:
# ── Config ────────────────────────────────────────────────────────────────────
ROOT = os.path.abspath('..')
with open(os.path.join(ROOT, 'config.json')) as f:
    cfg = json.load(f)

MODEL_NAME  = 'custom'   # 'custom' or 'vgg16'
DATA_DIR    = os.path.join(ROOT, cfg.get('data_dir', 'data'))
RESULTS_DIR = os.path.join(ROOT, cfg.get('results_dir', 'outputs/results'), f'seq_eval_{MODEL_NAME}')
cfg['ckpt_dir'] = os.path.join(ROOT, cfg['ckpt_dir'])
INF_CFG     = cfg['inference']

In [4]:
# ── Load both splits ──────────────────────────────────────────────────────────
with open(f"{RESULTS_DIR}/correct.json") as f:
    correct_entries = json.load(f)
with open(f"{RESULTS_DIR}/wrong.json") as f:
    wrong_entries = json.load(f)

print(f"Correct: {len(correct_entries)}  |  Wrong: {len(wrong_entries)}")

Correct: 5761  |  Wrong: 7307


In [5]:
# ── Load model ────────────────────────────────────────────────────────────────
model, device = load_model(
    MODEL_NAME,
    ckpt_dir=cfg['ckpt_dir'],
    num_classes=cfg['num_classes'],
    custom_cfg=cfg.get('custom'),
)
print(f"Model loaded on {device}")

Model loaded on cuda


In [6]:
# ── Load GT boxes ─────────────────────────────────────────────────────────────
gt_boxes_map = build_gt_boxes_map(os.path.join(DATA_DIR, 'test', 'digitStruct.mat'))
print(f"Loaded GT boxes for {len(gt_boxes_map)} images")

Loaded GT boxes for 13068 images


In [7]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def _upscale(bgr, scale):
    h, w = bgr.shape[:2]
    rgb  = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return cv2.resize(rgb, (w * scale, h * scale), interpolation=cv2.INTER_NEAREST)

def _load(filename):
    img = cv2.imread(os.path.join(DATA_DIR, 'test', filename))
    if img is None:
        raise FileNotFoundError(f"Image not found: {filename}")
    return img

# Distinct colours for DBSCAN clusters (noise = grey)
CLUSTER_COLOURS = ['#e6194b','#3cb44b','#4363d8','#f58231','#911eb4',
                   '#42d4f4','#f032e6','#bfef45','#fabed4','#469990']

In [8]:
# ── GT/Pred visualizer ────────────────────────────────────────────────────────
def visualize(entry):
    plt.close('all')
    filename = entry['filename']
    gt_seq   = entry['gt']
    pred_seq = entry['pred']

    img_bgr = _load(filename)
    scale   = 4
    img_large = _upscale(img_bgr, scale)

    _, det_list, _ = run_image(img_bgr, model, device, INF_CFG)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    ax = axes[0]
    ax.imshow(img_large)
    for (lx, ty, bw, bh, lbl) in gt_boxes_map.get(filename, []):
        ax.add_patch(patches.Rectangle(
            (lx*scale, ty*scale), bw*scale, bh*scale,
            linewidth=2, edgecolor='blue', facecolor='none'))
        ax.text(lx*scale, ty*scale-3, str(0 if lbl==10 else lbl),
                color='blue', fontsize=10, fontweight='bold')
    ax.set_title(f'Ground Truth: "{gt_seq}"', fontsize=12); ax.axis('off')

    ax = axes[1]
    ax.imshow(img_large)
    for (dx, dy, dw, dh, digit, conf) in det_list:
        ax.add_patch(patches.Rectangle(
            (dx*scale, dy*scale), dw*scale, dh*scale,
            linewidth=2, edgecolor='yellow', facecolor='none'))
        ax.text(dx*scale, dy*scale-3, digit,
                color='yellow', fontsize=10, fontweight='bold')
    ax.set_title(f'Predicted: "{pred_seq}"', fontsize=12); ax.axis('off')

    fig.suptitle(filename, fontsize=10, color='gray')
    plt.tight_layout(); plt.show()


# ── Pipeline visualizer (3×3 grid) ───────────────────────────────────────────
def visualize_pipeline(entry):
    plt.close('all')
    filename = entry['filename']
    gt_seq   = entry['gt']

    img_bgr = _load(filename)
    pred_seq, det_list, im = run_image(img_bgr, model, device, INF_CFG)

    scale = 4
    s3    = _upscale(im['step3'], scale)

    fig, axes = plt.subplots(3, 3, figsize=(20, 15))

    # Row 0: preprocessing
    for ax, (img, title) in zip(axes[0], [
        (_upscale(img_bgr,     scale), 'Original'),
        (_upscale(im['step1'], scale), 'Step 1: Polarity norm'),
        (_upscale(im['step2'], scale), 'Step 2: Gaussian blur'),
    ]):
        ax.imshow(img); ax.set_title(title, fontsize=10); ax.axis('off')

    # Row 1: proposals → classifier hits
    # Step 3: CLAHE
    axes[1][0].imshow(_upscale(im['step3'], scale))
    axes[1][0].set_title('Step 3: CLAHE', fontsize=10); axes[1][0].axis('off')

    # Step 4: MSER raw
    axes[1][1].imshow(s3)
    for (x, y, bw, bh) in im['mser_boxes']:
        axes[1][1].add_patch(patches.Rectangle((x*scale, y*scale), bw*scale, bh*scale,
                             linewidth=1, edgecolor='cyan', facecolor='none'))
    axes[1][1].set_title(f"Step 4: MSER ({len(im['mser_boxes'])})", fontsize=10); axes[1][1].axis('off')

    # Step 5: deduped proposals
    axes[1][2].imshow(s3)
    for (x, y, bw, bh) in im['all_proposals']:
        axes[1][2].add_patch(patches.Rectangle((x*scale, y*scale), bw*scale, bh*scale,
                             linewidth=1, edgecolor='orange', facecolor='none'))
    axes[1][2].set_title(f"Step 5: Deduped ({len(im['all_proposals'])})", fontsize=10); axes[1][2].axis('off')

    # Row 2: classifier hits → DBSCAN → final
    # Step 6: classifier hits (pre-NMS)
    axes[2][0].imshow(s3)
    for (x, y, bw, bh), lbl, conf in zip(im['digit_boxes'], im['digit_labels'], im['digit_scores']):
        axes[2][0].add_patch(patches.Rectangle((x*scale, y*scale), bw*scale, bh*scale,
                             linewidth=1, edgecolor='red', facecolor='none'))
        axes[2][0].text(x*scale, y*scale-2, f'{DIGIT_LABELS[lbl]}:{conf:.2f}', color='red', fontsize=6)
    axes[2][0].set_title(f"Step 6: Classifier hits ({len(im['digit_boxes'])})", fontsize=10); axes[2][0].axis('off')

    # Step 7: DBSCAN clusters on pre-NMS hits
    axes[2][1].imshow(s3)
    db_labels  = im['dbscan_labels']
    db_centers = im['dbscan_centers']
    if db_labels is not None:
        for (x, y, bw, bh), cl in zip(im['digit_boxes'], db_labels):
            colour = CLUSTER_COLOURS[int(cl) % len(CLUSTER_COLOURS)] if cl >= 0 else '#aaaaaa'
            axes[2][1].add_patch(patches.Rectangle((x*scale, y*scale), bw*scale, bh*scale,
                                 linewidth=2, edgecolor=colour, facecolor='none'))
        # Mark surviving cluster centres
        for cid, center in db_centers.items():
            colour = CLUSTER_COLOURS[int(cid) % len(CLUSTER_COLOURS)]
            axes[2][1].plot(center[0]*scale, center[1]*scale, '+', color=colour, markersize=12, markeredgewidth=2)
        n_strong = len(db_centers)
        n_noise  = int((db_labels == -1).sum())
        axes[2][1].set_title(f'Step 7: DBSCAN ({n_strong} strong clusters, {n_noise} noise)', fontsize=10)
    else:
        axes[2][1].set_title('Step 7: DBSCAN (n/a)', fontsize=10)
    axes[2][1].axis('off')

    # Step 8: final prediction
    axes[2][2].imshow(s3)
    for (x, y, bw, bh, digit, conf) in det_list:
        axes[2][2].add_patch(patches.Rectangle((x*scale, y*scale), bw*scale, bh*scale,
                             linewidth=2, edgecolor='yellow', facecolor='none'))
        axes[2][2].text(x*scale, y*scale-3, digit, color='yellow', fontsize=10, fontweight='bold')
    axes[2][2].set_title(f'Step 8: Prediction: "{pred_seq}" | GT: "{gt_seq}"', fontsize=10); axes[2][2].axis('off')

    fig.suptitle(filename, fontsize=11, color='gray')
    plt.tight_layout(); plt.show()

In [9]:
# ── GT/Pred: CORRECT entries ──────────────────────────────────────────────────
def show_correct(correct_idx):
    visualize(correct_entries[correct_idx])

widgets.interact(show_correct,
    correct_idx=widgets.IntSlider(value=0, min=0, max=len(correct_entries)-1, description='Correct'))

interactive(children=(IntSlider(value=0, description='Correct', max=5760), Output()), _dom_classes=('widget-in…

<function __main__.show_correct(correct_idx)>

In [10]:
# ── GT/Pred: WRONG entries ────────────────────────────────────────────────────
def show_wrong(wrong_idx):
    visualize(wrong_entries[wrong_idx])

widgets.interact(show_wrong,
    wrong_idx=widgets.IntSlider(value=0, min=0, max=len(wrong_entries)-1, description='Wrong'))

interactive(children=(IntSlider(value=0, description='Wrong', max=7306), Output()), _dom_classes=('widget-inte…

<function __main__.show_wrong(wrong_idx)>

In [11]:
# ── Pipeline: CORRECT entries ─────────────────────────────────────────────────
def show_pipeline_correct(pipe_correct_idx):
    visualize_pipeline(correct_entries[pipe_correct_idx])

widgets.interact(show_pipeline_correct,
    pipe_correct_idx=widgets.IntSlider(value=0, min=0, max=len(correct_entries)-1, description='Correct'))

interactive(children=(IntSlider(value=0, description='Correct', max=5760), Output()), _dom_classes=('widget-in…

<function __main__.show_pipeline_correct(pipe_correct_idx)>

In [13]:
# ── Pipeline: WRONG entries ───────────────────────────────────────────────────
def show_pipeline_wrong(pipe_wrong_idx):
    visualize_pipeline(wrong_entries[pipe_wrong_idx])

widgets.interact(show_pipeline_wrong,
    pipe_wrong_idx=widgets.IntSlider(value=0, min=0, max=len(wrong_entries)-1, description='Wrong'))

interactive(children=(IntSlider(value=0, description='Wrong', max=7306), Output()), _dom_classes=('widget-inte…

<function __main__.show_pipeline_wrong(pipe_wrong_idx)>